# CAP CDSS Benchmark Harness

**Automated evaluation of the CAP Clinical Decision Support pipeline.**

> This notebook is generated by `_generate_benchmark_evaluation_notebook.py`. Do not edit manually.

### Evaluators (9 metrics)

| Metric | What it measures |
|--------|-----------------|
| `severity_accuracy` | CURB65 severity tier exact match |
| `antibiotic_concordance` | First-line antibiotic substring match |
| `contradiction_recall` | Fraction of expected contradiction rules detected |
| `contradiction_precision` | Fraction of detected contradictions that are expected |
| `cxr_consolidation` | CXR consolidation detection accuracy |
| `cxr_localization` | Bounding box IoU vs RSNA annotations |
| `safety_score` | Safety rule compliance |
| `completeness` | Pipeline section coverage (8 sections) |

### Quick Start
1. Set runtime to **GPU -> A100** (Runtime -> Change runtime type)
2. Add **HF_TOKEN** to Colab Secrets (key icon in left sidebar)
3. *(Optional)* Add **LANGSMITH_API_KEY** for LangSmith tracing
4. **Run All** (Runtime -> Run all)


> **Disclaimer:** This notebook demonstrates a research prototype. All clinical outputs
> (severity scores, antibiotic recommendations, contradiction alerts, CXR analysis)
> are **AI-generated by MedGemma 1.5 4B** and have been validated only on synthetic data.
> This system is **not a medical device**, is not approved for clinical use, and must not
> be used for patient care decisions. See [DISCLAIMER.md](../DISCLAIMER.md) for full terms.


In [ ]:
# Cell 1: Install package + dependencies
import os

# Detect Colab vs local
try:
    from google.colab import userdata
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    # Install from GitHub (requires GITHUB_TOKEN in Colab Secrets)
    try:
        github_token = userdata.get("GITHUB_TOKEN")
        repo_url = f"git+https://{github_token}@github.com/HP-00/CAP-CDSS-MedGemma.git@main"
    except Exception:
        repo_url = "git+https://github.com/HP-00/CAP-CDSS-MedGemma.git@main"

    %pip install --quiet {repo_url}
    %pip install --quiet plotly>=5.0.0 pandas>=2.0.0 langsmith>=0.1.0 nest-asyncio
else:
    print("Local environment detected. Ensure: pip install -e '.[dev,benchmark]'")

import nest_asyncio
nest_asyncio.apply()

print("Setup complete")


## Authentication


In [ ]:
import os

# HuggingFace token for gated model access
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except ImportError:
    pass

assert os.environ.get("HF_TOKEN"), "HF_TOKEN not set! Add it to Colab Secrets or environment."

# Optional: LangSmith tracing
try:
    from google.colab import userdata as _ud
    langsmith_key = _ud.get("LANGSMITH_API_KEY")
    if langsmith_key:
        os.environ["LANGSMITH_API_KEY"] = langsmith_key
        os.environ["LANGSMITH_TRACING"] = "true"
        os.environ["LANGSMITH_PROJECT"] = "cap-benchmark"
        print(f"LangSmith tracing enabled (project: cap-benchmark)")
    else:
        print("LANGSMITH_API_KEY not found - tracing disabled (optional)")
except Exception:
    print("LangSmith tracing disabled (optional)")

print("Authentication complete")


## Download & Locate CXR Images

Google CXR pair for the T=0 builtin case + 12 RSNA images from the installed package.


In [ ]:
import os, glob, urllib.request

# 1. Google CXR pair for T=0 builtin case
CXR_BASE = "https://storage.googleapis.com/hai-cd3-foundations-public-vault-entry/med_gemma/colab_example/cxr"
CXR_FILES = {
    "longitudinal_cxr_after.png": f"{CXR_BASE}/longitudinal_cxr_after.png",
    "longitudinal_cxr_before.png": f"{CXR_BASE}/longitudinal_cxr_before.png",
}
for filename, url in CXR_FILES.items():
    if not os.path.exists(filename):
        print(f"Downloading {filename}...")
        urllib.request.urlretrieve(url, filename)
    else:
        print(f"Already exists: {filename}")
CXR_CURRENT = "longitudinal_cxr_after.png"
CXR_PRIOR = "longitudinal_cxr_before.png"

# 2. Locate RSNA images from installed package
import benchmark_data.rsna
RSNA_IMG_DIR = os.path.join(os.path.dirname(benchmark_data.rsna.__file__), "images")
rsna_images = sorted(glob.glob(os.path.join(RSNA_IMG_DIR, "*.png")))
print(f"Google CXR: 2 images ready")
print(f"RSNA CXR: {len(rsna_images)} images in {RSNA_IMG_DIR}")


## Load Model & Build Graph


In [ ]:
import time
from cap_agent.models.medgemma import get_model_and_processor
from cap_agent.agent.graph import build_cap_agent_graph

# Load MedGemma (lazy singleton - only loads once, includes warm-up)
model, processor = get_model_and_processor()
print(f"Model loaded on {model.device}")

# Build the 8-node LangGraph pipeline
graph = build_cap_agent_graph()
print("Graph compiled: 8 nodes, conditional routing at contradiction check")


## Baseline Metrics

Embedded baseline from the 2026-02-22 Run 2 GPU run (49 cases on A100). Used for regression detection.


In [ ]:
# Baseline metrics (from 2026-02-22 Run 2 GPU run, 49 cases on A100)
# These are the scores to beat — any regression below these triggers a warning
BASELINE_METRICS = {
    "severity_accuracy": 1.0,
    "antibiotic_concordance": 1.0,
    "safety_score": 1.0,
    "completeness": 1.0,
    "contradiction_recall": 0.898,
    "contradiction_precision": 0.898,
    "cxr_consolidation": 0.5833,
    "cxr_localization": 0.1829,
}

# Per-metric regression thresholds
# Deterministic metrics have zero tolerance; stochastic allow small variance
REGRESSION_THRESHOLDS = {
    "severity_accuracy": 0.0,
    "antibiotic_concordance": 0.0,
    "contradiction_recall": 0.0,
    "safety_score": 0.0,
    "completeness": 0.0,
    "contradiction_precision": 0.05,
    "cxr_consolidation": 0.05,
    "cxr_localization": 0.10,
}

print("Baseline metrics loaded:")
for metric, val in sorted(BASELINE_METRICS.items()):
    threshold = REGRESSION_THRESHOLDS.get(metric, 0.0)
    print(f"  {metric:30s}  {val:.2%}  (tolerance: {threshold:.0%})")


## Run Full Benchmark

Runs all 49 benchmark cases through the real MedGemma pipeline:
- **4 builtin cases:** T=0 (FHIR+CXR), T=48h (stewardship), CR-10 (safety), Day 3-4 (monitoring)
- **33 Track 2 cases:** CURB65 boundaries, contradiction rules, stewardship modifiers
- **12 Track 1 cases:** Real RSNA CXR images with bounding box ground truth

Estimated time: ~100 minutes on A100 GPU.


In [ ]:
from benchmark_data.evaluation.run_benchmark import run_benchmark, get_builtin_cases
from benchmark_data.cases.registry import get_track2_cases, get_track1_cases
import copy

# 1. Builtin cases (4): T=0, T=48h, CR-10, Day 3-4
builtin_cases = get_builtin_cases()
# Inject Google CXR into T=0
builtin_cases[0]["cxr"]["image_path"] = CXR_CURRENT
builtin_cases[0]["cxr"]["prior_image_path"] = CXR_PRIOR

# 2. Track 2 synthetic cases (33): CURB65 boundaries, contradictions, stewardship
track2_cases = get_track2_cases()

# 3. Track 1 RSNA cases (12): real CXR images with bounding box GT
track1_cases = copy.deepcopy(get_track1_cases())
# Fix image paths to absolute paths from installed package
for case in track1_cases:
    img_name = os.path.basename(case["cxr"]["image_path"])
    case["cxr"]["image_path"] = os.path.join(RSNA_IMG_DIR, img_name)
    if case["cxr"].get("prior_image_path"):
        prior_name = os.path.basename(case["cxr"]["prior_image_path"])
        case["cxr"]["prior_image_path"] = os.path.join(RSNA_IMG_DIR, prior_name)

cases = builtin_cases + track2_cases + track1_cases
print(f"Prepared {len(cases)} cases: {len(builtin_cases)} builtin + {len(track2_cases)} Track 2 + {len(track1_cases)} Track 1")

# Run full benchmark
benchmark_start = time.time()
benchmark_result = run_benchmark(mode="full", cases=cases, output_dir="benchmark_results")
benchmark_elapsed = time.time() - benchmark_start

metrics = benchmark_result["metrics"]
regressions = benchmark_result["regressions"]
all_results = benchmark_result["results"]
chart_data = benchmark_result["chart_data"]

print(f"\nBenchmark complete in {benchmark_elapsed:.1f}s")
print(f"Cases: {len(all_results)}, Metrics: {len(metrics)}")
errors = sum(1 for r in all_results if r.get("error"))
if errors:
    print(f"Errors: {errors}")


In [ ]:
# Generate dynamic labels for all cases
_BUILTIN_LABELS = {
    0: "T=0 (FHIR+CXR)",
    1: "T=48h (Stewardship)",
    2: "CR-10 (Safety)",
    3: "Day 3-4 (Monitoring)",
}

def _case_label(case, i):
    """Generate a display label for a benchmark case."""
    if i in _BUILTIN_LABELS and i < len(builtin_cases):
        return _BUILTIN_LABELS[i]
    case_id = case.get("case_id", "") if isinstance(case, dict) else ""
    if case_id.startswith("T1-"):
        return f"T1: {case_id}"
    if "description" in case:
        return case["description"][:40]
    return case_id or f"Case {i}"

case_labels = [_case_label(c, i) for i, c in enumerate(cases)]
print(f"Generated {len(case_labels)} case labels")


## Aggregate Metrics


In [ ]:
ESC = chr(27)
B = f"{ESC}[1m"; G = f"{ESC}[92m"; R = f"{ESC}[91m"; Y = f"{ESC}[93m"
C = f"{ESC}[96m"; X = f"{ESC}[0m"

print(f"\n{B}{C}{'=' * 70}")
print(f"  BENCHMARK METRICS")
print(f"{'=' * 70}{X}\n")

header = f"  {'Metric':<30} {'Mean':>8} {'Min':>8} {'Max':>8}  {'Status'}"
print(header)
print(f"  {'─' * 66}")

for name, vals in sorted(metrics.items()):
    mean = vals["mean"]
    mn = vals["min"]
    mx = vals["max"]

    baseline_val = BASELINE_METRICS.get(name)
    threshold = REGRESSION_THRESHOLDS.get(name, 0.0)

    if baseline_val is not None and mean < baseline_val - threshold:
        status = f"{R}REGRESSION{X}"
    elif mean >= 0.95:
        status = f"{G}EXCELLENT{X}"
    elif mean >= 0.7:
        status = f"{Y}GOOD{X}"
    else:
        status = f"{R}NEEDS WORK{X}"

    print(f"  {name:<30} {mean:>7.2%} {mn:>7.2%} {mx:>7.2%}  {status}")

print(f"\n{C}{'=' * 70}{X}")


## Capability Radar

Current performance vs baseline overlay across 5 capability axes.


In [ ]:
import plotly.graph_objects as go

# Current capability axes from chart_data
current_axes = chart_data.get("capability_axes", {})

# Baseline axes
baseline_axes = {
    "Extraction": 1.0,
    "Guideline Adherence": 1.0,
    "Safety": 1.0,
    "Severity Scoring": 0.5,
    "Contradiction Detection": 0.875,
}

if current_axes:
    categories = list(current_axes.keys())
    current_vals = [current_axes[c] for c in categories]
    baseline_vals = [baseline_axes.get(c, 0) for c in categories]
    # Close the polygon
    categories_closed = categories + [categories[0]]
    current_closed = current_vals + [current_vals[0]]
    baseline_closed = baseline_vals + [baseline_vals[0]]

    fig = go.Figure()
    fig.add_trace(go.Scatterpolar(
        r=current_closed, theta=categories_closed,
        fill="toself", name="Current", line_color="#2196F3",
    ))
    fig.add_trace(go.Scatterpolar(
        r=baseline_closed, theta=categories_closed,
        fill="toself", name="Baseline", line_color="#FF9800", opacity=0.4,
    ))
    fig.update_layout(
        title="Capability Radar: Current vs Baseline",
        polar=dict(radialaxis=dict(visible=True, range=[0, 1])),
        width=600, height=500,
    )
    fig.show()
else:
    print("No capability axes data available")


## Per-Case Results


In [ ]:
ESC = chr(27)
B = f"{ESC}[1m"; G = f"{ESC}[92m"; R = f"{ESC}[91m"; Y = f"{ESC}[93m"
C = f"{ESC}[96m"; X = f"{ESC}[0m"

print(f"\n{B}{C}{'=' * 70}")
print(f"  PER-CASE RESULTS")
print(f"{'=' * 70}{X}\n")

for i, result in enumerate(all_results):
    label = case_labels[i] if i < len(case_labels) else f"Case {i}"
    case_id = result.get("case_id", "unknown")
    elapsed = result.get("elapsed_seconds", 0)
    error = result.get("error")
    scores = result.get("scores", {})

    status_color = R if error else G
    status_str = f"{R}ERROR: {error}{X}" if error else f"{G}OK{X}"

    print(f"{B}{label}{X} ({case_id})")
    print(f"  Time: {elapsed:.3f}s  Status: {status_str}")

    if scores:
        for metric, score in sorted(scores.items()):
            sc = G if score >= 0.95 else (Y if score >= 0.5 else R)
            print(f"    {metric:<30} {sc}{score:.2%}{X}")
    print()


## Individual Evaluator Analysis


In [ ]:
import plotly.graph_objects as go

# Severity accuracy per case
severity_preds = chart_data.get("severity_predictions", [])

if severity_preds:
    from benchmark_data.evaluation.generate_report import build_severity_confusion_matrix
    fig = build_severity_confusion_matrix(severity_preds)
    fig.show()

# Per-case severity details
print("\nSeverity scoring per case:")
for i, result in enumerate(all_results):
    output = result.get("pipeline_output")
    if not output:
        continue
    curb = output.get("curb65_score", {})
    label = case_labels[i] if i < len(case_labels) else f"Case {i}"
    gt = cases[i].get("ground_truth", {}) if i < len(cases) else {}
    expected = gt.get("severity_tier", "?")
    actual = curb.get("severity_tier", "?")
    match = "CORRECT" if expected == actual else "WRONG"
    print(f"  {label}: CURB65={curb.get('curb65', '?')} {actual} (expected: {expected}) — {match}")


In [ ]:
print("Antibiotic concordance per case:")
for i, result in enumerate(all_results):
    output = result.get("pipeline_output")
    if not output:
        continue
    abx = output.get("antibiotic_recommendation", {})
    label = case_labels[i] if i < len(case_labels) else f"Case {i}"
    gt = cases[i].get("ground_truth", {}) if i < len(cases) else {}
    expected = gt.get("antibiotic", "?")
    actual = abx.get("first_line", "N/A")
    match = "MATCH" if expected.lower() in actual.lower() else "MISMATCH"
    print(f"  {label}: {actual} (expected: {expected}) — {match}")


In [ ]:
import plotly.graph_objects as go

print("Contradiction detection per case:")
for i, result in enumerate(all_results):
    output = result.get("pipeline_output")
    if not output:
        continue
    label = case_labels[i] if i < len(case_labels) else f"Case {i}"
    gt = cases[i].get("ground_truth", {}) if i < len(cases) else {}
    expected = set(gt.get("contradictions", []))
    actual = {c["rule_id"] for c in output.get("contradictions_detected", [])}
    recall = len(actual & expected) / len(expected) if expected else 1.0
    precision = len(actual & expected) / len(actual) if actual else 1.0
    print(f"  {label}: expected={expected or 'none'} actual={actual or 'none'} "
          f"recall={recall:.0%} precision={precision:.0%}")

# Grouped bar chart
all_expected = set()
all_actual = set()
for i, result in enumerate(all_results):
    output = result.get("pipeline_output")
    if not output:
        continue
    gt = cases[i].get("ground_truth", {}) if i < len(cases) else {}
    all_expected |= set(gt.get("contradictions", []))
    all_actual |= {c["rule_id"] for c in output.get("contradictions_detected", [])}

all_rules = sorted(all_expected | all_actual)
if all_rules:
    recalls = []
    precisions = []
    for rule in all_rules:
        # Count across cases
        tp = sum(1 for i, r in enumerate(all_results)
                 if r.get("pipeline_output")
                 and rule in {c["rule_id"] for c in r["pipeline_output"].get("contradictions_detected", [])}
                 and rule in set(cases[i].get("ground_truth", {}).get("contradictions", [])))
        fn = sum(1 for i, _ in enumerate(all_results)
                 if rule in set(cases[i].get("ground_truth", {}).get("contradictions", []))
                 and (not all_results[i].get("pipeline_output")
                      or rule not in {c["rule_id"] for c in all_results[i]["pipeline_output"].get("contradictions_detected", [])}))
        fp = sum(1 for i, r in enumerate(all_results)
                 if r.get("pipeline_output")
                 and rule in {c["rule_id"] for c in r["pipeline_output"].get("contradictions_detected", [])}
                 and rule not in set(cases[i].get("ground_truth", {}).get("contradictions", [])))
        recalls.append(tp / (tp + fn) if (tp + fn) > 0 else 0)
        precisions.append(tp / (tp + fp) if (tp + fp) > 0 else 0)

    fig = go.Figure(data=[
        go.Bar(name="Recall", x=all_rules, y=recalls, marker_color="#2196F3"),
        go.Bar(name="Precision", x=all_rules, y=precisions, marker_color="#FF9800"),
    ])
    fig.update_layout(
        title="Contradiction Detection (per rule)",
        barmode="group",
        yaxis=dict(range=[0, 1.05], title="Score"),
        width=600, height=400,
    )
    fig.show()


In [ ]:
print("Safety score per case:")
for i, result in enumerate(all_results):
    scores = result.get("scores", {})
    label = case_labels[i] if i < len(case_labels) else f"Case {i}"
    safety = scores.get("safety_score", "N/A")
    if isinstance(safety, float):
        print(f"  {label}: {safety:.0%}")
    else:
        print(f"  {label}: {safety}")


In [ ]:
print("Completeness per case:")
for i, result in enumerate(all_results):
    output = result.get("pipeline_output")
    scores = result.get("scores", {})
    label = case_labels[i] if i < len(case_labels) else f"Case {i}"
    completeness = scores.get("completeness", "N/A")

    if isinstance(completeness, float):
        print(f"  {label}: {completeness:.0%}")
    else:
        print(f"  {label}: {completeness}")

    # Show section coverage
    if output:
        so = output.get("structured_output", {})
        sections = so.get("sections", so)
        found = [k for k in sorted(sections.keys()) if any(k.startswith(p) for p in ("1_", "2_", "3_", "4_", "5_", "6_", "7_", "8_"))]
        print(f"    Sections: {', '.join(found)}")


In [ ]:
print("CXR consolidation per case:")
for i, result in enumerate(all_results):
    scores = result.get("scores", {})
    label = case_labels[i] if i < len(case_labels) else f"Case {i}"
    cxr = scores.get("cxr_consolidation")
    if cxr is not None:
        print(f"  {label}: {cxr:.0%}")
    else:
        print(f"  {label}: skipped (no CXR ground truth)")


In [ ]:
ESC = chr(27)
G = f"{ESC}[92m"; R = f"{ESC}[91m"; Y = f"{ESC}[93m"; X = f"{ESC}[0m"

print("CXR localization (IoU) per case:")
for i, result in enumerate(all_results):
    scores = result.get("scores", {})
    label = case_labels[i] if i < len(case_labels) else f"Case {i}"
    iou = scores.get("cxr_localization")
    if iou is not None:
        color = G if iou >= 0.5 else (Y if iou >= 0.2 else R)
        print(f"  {label}: {color}{iou:.3f}{X}")
    else:
        print(f"  {label}: skipped (no bbox ground truth)")


In [ ]:
import plotly.graph_objects as go

latencies = [r["elapsed_seconds"] for r in all_results]
labels = case_labels[:len(all_results)]

fig = go.Figure(data=go.Bar(
    x=latencies, y=labels, orientation="h",
    marker_color="#2196F3",
    text=[f"{t:.2f}s" for t in latencies],
    textposition="auto",
))
fig.update_layout(
    title="Per-Case Pipeline Latency",
    xaxis_title="Time (seconds)",
    width=600, height=max(250, 60 * len(labels)),
)
fig.show()

total = sum(latencies)
mean = total / len(latencies) if latencies else 0
print(f"\nTotal: {total:.1f}s, Mean per case: {mean:.1f}s")


## Regression Check

Compare current results against the embedded baseline.


In [ ]:
from benchmark_data.evaluation.compare_runs import check_regression

ESC = chr(27)
B = f"{ESC}[1m"; G = f"{ESC}[92m"; R = f"{ESC}[91m"; Y = f"{ESC}[93m"
C = f"{ESC}[96m"; X = f"{ESC}[0m"

current_means = {k: v["mean"] for k, v in metrics.items()}
regressions = check_regression(current_means, BASELINE_METRICS)

print(f"\n{B}{C}{'=' * 70}")
print(f"  REGRESSION CHECK")
print(f"{'=' * 70}{X}\n")

header = f"  {'Metric':<30} {'Baseline':>10} {'Current':>10} {'Status':>12}"
print(header)
print(f"  {'─' * 62}")

regressed_metrics = {r["metric"] for r in regressions}
for metric in sorted(set(list(BASELINE_METRICS.keys()) + list(current_means.keys()))):
    baseline_val = BASELINE_METRICS.get(metric)
    current_val = current_means.get(metric)
    bl_str = f"{baseline_val:.2%}" if baseline_val is not None else "N/A"
    cur_str = f"{current_val:.2%}" if current_val is not None else "N/A"

    if metric in regressed_metrics:
        status = f"{R}REGRESSION{X}"
    elif current_val is not None and baseline_val is not None and current_val > baseline_val:
        status = f"{G}IMPROVED{X}"
    else:
        status = f"{G}PASS{X}"

    print(f"  {metric:<30} {bl_str:>10} {cur_str:>10}  {status}")

if regressions:
    print(f"\n{R}{B}REGRESSIONS DETECTED: {len(regressions)}{X}")
    for r in regressions:
        print(f"  {R}{r['metric']}: {r['baseline']:.4f} -> {r['current']:.4f} (degradation: {r['degradation']:.4f}){X}")
else:
    print(f"\n{G}{B}No regressions detected.{X}")


## Clinical Output Samples

Rendered clinical output for each successful benchmark case.


In [ ]:
# Clinical output renderer — formats structured_output as a readable document
def render_clinical_output(result, scenario_title="Pipeline Output"):
    """Render structured_output as a formatted clinical document using ANSI codes."""
    ESC = chr(27)
    B = f"{ESC}[1m"       # Bold
    R = f"{ESC}[91m"      # Red
    Y = f"{ESC}[93m"      # Yellow
    G = f"{ESC}[92m"      # Green
    C = f"{ESC}[96m"      # Cyan
    X = f"{ESC}[0m"       # Reset

    so = result.get("structured_output", {})
    sections = so.get("sections", {})

    print(f"\n{B}{C}{'=' * 70}")
    print(f"  CLINICAL DECISION SUPPORT — {scenario_title}")
    print(f"{'=' * 70}{X}")
    print(f"{Y}AI-generated observations for clinician review — not a substitute for clinical judgement.{X}\n")

    # 1. PATIENT
    s1 = sections.get("1_patient", {})
    demo = s1.get("demographics", {})
    print(f"{B}{C}1. PATIENT{X}")
    print(f"  {demo.get('age', '?')}yo {demo.get('sex', '?')}")
    allergy_list = demo.get("allergies", [])
    if allergy_list:
        for a in allergy_list:
            if isinstance(a, dict):
                print(f"  {R}Allergy: {a.get('drug', '?')} — {a.get('reaction_type', '?')} ({a.get('severity', '?')}){X}")
            else:
                print(f"  {R}Allergy: {a}{X}")
    else:
        print(f"  {G}No known drug allergies{X}")
    combos = demo.get("comorbidities", [])
    if combos:
        print(f"  Comorbidities: {', '.join(combos)}")
    if demo.get("pregnancy"):
        print(f"  {R}PREGNANT{X}")
    print(f"  Oral tolerance: {'Yes' if demo.get('oral_tolerance', True) else R + 'No' + X}")
    travel = demo.get("travel_history", [])
    if travel:
        print(f"  {Y}Travel: {', '.join(travel)}{X}")
    print()

    # 2. SEVERITY
    s2 = sections.get("2_severity", {})
    curb = s2.get("curb65", {})
    sev = curb.get("severity_tier", "?")
    sev_color = R if sev == "high" else (Y if sev == "moderate" else G)
    score = curb.get("curb65")
    score_str = f"CURB65={score}" if score is not None else f"CRB65={curb.get('crb65', '?')} (urea unavailable)"
    print(f"{B}{C}2. SEVERITY{X}")
    print(f"  {score_str}  {sev_color}{B}{sev.upper()}{X}")
    print(f"  C={curb.get('c','?')} U={curb.get('u','?')} R={curb.get('r','?')} B={curb.get('b','?')} 65={curb.get('age_65','?')}")
    missing = curb.get("missing_variables", [])
    if missing:
        print(f"  {Y}Missing: {', '.join(missing)}{X}")
    print()

    # 3. CXR
    s3 = sections.get("3_cxr", {})
    cxr = s3.get("findings", {})
    print(f"{B}{C}3. CHEST X-RAY{X}")
    for finding in ["consolidation", "pleural_effusion", "cardiomegaly", "edema", "atelectasis"]:
        f = cxr.get(finding, {})
        if not isinstance(f, dict):
            continue
        present = f.get("present", False)
        conf = f.get("confidence", "?")
        if present:
            loc = f.get("location", "")
            bbox = f.get("bounding_box")
            line = f"  {R}PRESENT{X} {finding} ({conf} confidence)"
            if loc:
                line += f" at {loc}"
            if bbox:
                line += f" [bbox: {bbox}]"
            print(line)
        else:
            print(f"  {G}absent{X}  {finding} ({conf} confidence)")
    iq = cxr.get("image_quality", {})
    if iq:
        print(f"  Quality: {iq.get('projection','?')} / {iq.get('rotation','?')} / {iq.get('penetration','?')}")
    longit = cxr.get("longitudinal_comparison")
    if longit:
        print(f"  {B}Longitudinal:{X}")
        for fn, ch in longit.items():
            if isinstance(ch, dict):
                print(f"    {fn}: {ch.get('change','?')} — {ch.get('description','')}")
    print()

    # 4. KEY BLOODS
    s4 = sections.get("4_key_bloods", {})
    labs = s4.get("values", {})
    print(f"{B}{C}4. KEY BLOODS{X}")
    for test, data in (labs or {}).items():
        if not isinstance(data, dict):
            continue
        val = data.get("value", "?")
        unit = data.get("unit", "")
        ref = data.get("reference_range", "")
        abn = data.get("abnormal_flag", False)
        color = R if abn else G
        flag = " ABNORMAL" if abn else ""
        print(f"  {color}{test}: {val} {unit}{flag}{X}  (ref: {ref})")
    print()

    # 5. CONTRADICTIONS
    s5 = sections.get("5_contradiction_alert", {})
    alerts = s5.get("alerts", [])
    informational = s5.get("informational", [])
    resolutions = s5.get("resolutions", [])
    n_total = s5.get("detected", 0)
    print(f"{B}{C}5. CONTRADICTION ALERTS{X}")
    if n_total == 0:
        print(f"  {G}No contradictions detected — findings concordant{X}")
    else:
        print(f"  {n_total} contradiction(s) detected")
        for c in alerts:
            conf = c.get("confidence", "?")
            badge_color = R if conf == "high" else Y
            print(f"  {badge_color}[{conf.upper()}]{X} {c.get('rule_id','?')}: {c.get('pattern','')}")
            if c.get("evidence_for"):
                print(f"    For: {c['evidence_for']}")
            if c.get("evidence_against"):
                print(f"    Against: {c['evidence_against']}")
            rec = c.get("recommendation")
            if rec:
                print(f"    {B}Recommendation:{X} {rec}")
        for c in informational:
            print(f"  {G}[LOW]{X} {c.get('rule_id','?')}: {c.get('pattern','')}")
        if resolutions:
            print(f"  {B}Resolutions:{X}")
            for r in resolutions:
                print(f"    {r[:200]}")
    print()

    # 6. TREATMENT
    s6 = sections.get("6_treatment_pathway", {})
    abx = s6.get("antibiotic", {})
    print(f"{B}{C}6. TREATMENT{X}")
    print(f"  First-line: {abx.get('first_line', 'N/A')}")
    print(f"  Dose/route: {abx.get('dose_route', 'N/A')}")
    if abx.get("allergy_adjustment"):
        print(f"  {Y}Allergy adj: {abx['allergy_adjustment']}{X}")
    if abx.get("atypical_cover"):
        print(f"  Atypical: {abx['atypical_cover']}")
    if abx.get("renal_adjustment"):
        print(f"  {Y}Renal: {abx['renal_adjustment']}{X}")
    cortico = s6.get("corticosteroid")
    if cortico:
        print(f"  Corticosteroid: {cortico}")
    steward = abx.get("stewardship_notes", [])
    for note in steward:
        print(f"  {Y}Stewardship: {note}{X}")
    inv = s6.get("investigations", {})
    if inv:
        print(f"  {B}Investigations:{X}")
        for name, detail in inv.items():
            if isinstance(detail, dict):
                rec = "Recommended" if detail.get("recommended") else "Not indicated"
                print(f"    {name}: {rec} — {detail.get('reasoning', '')[:100]}")
    print()

    # 7. DATA GAPS
    s7 = sections.get("7_data_gaps", {})
    gaps = s7.get("gaps", [])
    print(f"{B}{C}7. DATA GAPS{X}")
    if gaps:
        for g in gaps:
            print(f"  {Y}- {g}{X}")
    else:
        print(f"  {G}None identified{X}")
    print()

    # 8. MONITORING
    s8 = sections.get("8_monitoring", {})
    plan = s8.get("plan", {})
    print(f"{B}{C}8. MONITORING{X}")
    crp_trend = plan.get("crp_trend")
    if crp_trend:
        adm = crp_trend.get("admission_value", "?")
        cur = crp_trend.get("current_value", "?")
        pct = crp_trend.get("percent_change", "?")
        trend = crp_trend.get("trend", "?")
        sr = crp_trend.get("flag_senior_review", False)
        sr_color = R if sr else G
        print(f"  CRP: {adm} -> {cur} mg/L ({pct}% change, {trend})")
        print(f"  Senior review: {sr_color}{sr}{X}")
    tr = plan.get("treatment_response")
    if tr:
        reassess = tr.get("reassess_needed", False)
        ra_color = R if reassess else G
        print(f"  Treatment response: {ra_color}reassess_needed={reassess}{X}")
        for action in tr.get("actions", []):
            print(f"    - {action}")
    dc = plan.get("discharge_criteria_met")
    if dc is not None:
        dc_color = G if dc else R
        dc_str = "MET" if dc else "NOT MET"
        print(f"  Discharge: {dc_color}{dc_str}{X}")
    dc_detail = plan.get("discharge_criteria_details", {})
    if dc_detail:
        for check, passed_val in dc_detail.items():
            if check.startswith("_"):
                continue
            chk_color = G if passed_val else R
            chk_str = "PASS" if passed_val else "FAIL"
            print(f"    {chk_color}[{chk_str}]{X} {check}")
    print(f"  Next review: {plan.get('next_review', 'N/A')}")
    print()

    # PROVENANCE
    prov = so.get("provenance", {})
    if prov:
        print(f"{B}{C}PROVENANCE{X}")
        tools = prov.get("extraction_tools", {})
        sources = prov.get("data_sources", {})
        for tool_name, pipeline in tools.items():
            src = sources.get(tool_name, [])
            print(f"  {tool_name}: {pipeline} ({', '.join(src) if src else 'N/A'})")

    print(f"\n{C}{'=' * 70}{X}\n")

print("render_clinical_output() defined")


In [ ]:
for i, result in enumerate(all_results):
    output = result.get("pipeline_output")
    if not output:
        print(f"Case {i}: skipped (pipeline error)")
        continue
    label = case_labels[i] if i < len(case_labels) else f"Case {i}"
    render_clinical_output(output, label)


## HTML Report

Self-contained HTML report with interactive Plotly charts.


In [ ]:
from benchmark_data.evaluation.generate_report import generate_html_report
import IPython.display

report_path = "benchmark_results/report.html"
generate_html_report(metrics, chart_data, report_path)
print(f"Report generated: {report_path}")

# Display inline
with open(report_path) as f:
    report_html = f.read()
IPython.display.display(IPython.display.HTML(report_html))


## Download Results

Save metrics JSON and HTML report for local analysis.


In [ ]:
import json

# Save metrics summary
metrics_path = "benchmark_results/metrics.json"
with open(metrics_path) as f:
    saved_data = json.load(f)

print(f"Metrics saved: {metrics_path}")
print(f"Report saved: benchmark_results/report.html")
print(f"\nMetrics summary:")
for name, vals in sorted(saved_data.get("metrics", {}).items()):
    print(f"  {name}: {vals}")

# Download files in Colab
try:
    from google.colab import files
    files.download(metrics_path)
    files.download("benchmark_results/report.html")
    print("\nDownload started!")
except ImportError:
    print("\nNot in Colab — files saved locally")


## Summary

Overall benchmark verdict.


In [ ]:
ESC = chr(27)
B = f"{ESC}[1m"; G = f"{ESC}[92m"; R = f"{ESC}[91m"; Y = f"{ESC}[93m"
C = f"{ESC}[96m"; X = f"{ESC}[0m"

# Compute verdict
n_cases = len(all_results)
n_errors = sum(1 for r in all_results if r.get("error"))
n_passed = n_cases - n_errors
n_regressions = len(regressions)
total_time = sum(r.get("elapsed_seconds", 0) for r in all_results)

# Overall pass/fail
all_pass = n_errors == 0 and n_regressions == 0

print(f"\n{B}{C}{'=' * 70}")
print(f"  BENCHMARK SUMMARY")
print(f"{'=' * 70}{X}\n")

verdict_color = G if all_pass else R
verdict = "PASS" if all_pass else "FAIL"
print(f"  {B}Verdict: {verdict_color}{verdict}{X}")
print()
print(f"  Cases: {n_passed}/{n_cases} passed")
print(f"  Errors: {n_errors}")
print(f"  Regressions: {n_regressions}")
print(f"  Total time: {total_time:.1f}s ({total_time/n_cases:.1f}s per case)")
print()

# Metric summary
print(f"  {B}Metrics:{X}")
for name, vals in sorted(metrics.items()):
    mean = vals["mean"]
    color = G if mean >= 0.95 else (Y if mean >= 0.7 else R)
    print(f"    {name:<30} {color}{mean:.2%}{X}")

print(f"\n{C}{'=' * 70}{X}")
if all_pass:
    print(f"\n{G}{B}All checks passed! Pipeline performing at or above baseline.{X}")
else:
    print(f"\n{R}{B}Issues detected — review regressions and errors above.{X}")
